In [2]:
from datasets import load_dataset

dataset_id = "AI-MO/NuminaMath-TIR"
train_dataset, test_dataset = load_dataset(
    dataset_id, split=["train[:5%]", "test[:5%]"]
)

/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/multiprocess/connection.py:335: SyntaxWarning: 'return' in a 'finally' block
  return f
/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/multiprocess/connection.py:337: SyntaxWarning: 'return' in a 'finally' block
  return self._get_more_data(ov, maxsize)


TypeError: Pickler._batch_setitems() takes 2 positional arguments but 3 were given

In [ ]:
print(train_dataset[10])


{'problem': 'Given that the function $y=f\\left(x\\right)$ is an odd function, defined on $R$, and satisfies $f\\left(x+4\\right)=f\\left(x\\right)$. When $x\\in \\left(0,2\\right)$, $f\\left(x\\right)=2^{x}+\\log _{3}(x+2)$, find $f\\left(2023\\right)$.', 'solution': 'To solve the problem, let\'s break it down step-by-step:\n\n1. **Understand the properties of the function \\( f(x) \\)**:\n   Given that \\( f(x) \\) is an odd function and periodic with period 4, i.e., \\( f(x+4) = f(x) \\).\n   \n2. **Analyze the function in the interval \\( (0, 2) \\)**:\n   Given \\( f(x) = 2^x + \\log_3(x+2) \\) for \\( x \\in (0, 2) \\).\n\n3. **Find \\( f(2023) \\)**:\n   Use the periodic property to reduce the computation.\n\nSince \\( f(x) \\) is periodic with period 4, we can reduce \\( x = 2023 \\) modulo 4 to find the corresponding \\( x \\) in the interval \\( [-2, 2] \\).\n\n4. **Implementation in Python**:\n   Determine \\( 2023 \\) modulo 4, and then use the given formula to find \\( f(2

In [ ]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)


def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }


train_dataset = train_dataset.map(make_conversation)
test_dataset = test_dataset.map(make_conversation)

In [ ]:
print(train_dataset[0]["prompt"])

[{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer>', 'role': 'system'}, {'content': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'role': 'user'}]


In [ ]:
train_dataset = train_dataset.remove_columns(["messages", "problem"])
print(train_dataset)

Dataset({
    features: ['solution', 'prompt'],
    num_rows: 3622
})


In [3]:
from transformers import AutoModelForCausalLM

model_id = "Qwen/Qwen2-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)

W0312 12:30:02.252000 73222 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/torchao/quantization/quant_api.py:1825: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
import re


def format_reward(completions, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content) for content in completion_contents]
    return [1.0 if match else 0.0 for match in matches]

In [ ]:
from math_verify import LatexExtractionConfig, parse, verify


def accuracy_reward(completions, **kwargs):
    """Reward function that checks if the completion is the same as the ground truth."""
    solutions = kwargs["solution"]
    completion_contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, solution in zip(completion_contents, solutions):
        gold_parsed = parse(
            solution,
            extraction_mode="first_match",
            extraction_config=[LatexExtractionConfig()],
        )
        answer_parsed = parse(
            content,
            extraction_mode="first_match",
            extraction_config=[LatexExtractionConfig()],
        )
        if len(gold_parsed) != 0:
            try:
                rewards.append(float(verify(answer_parsed, gold_parsed)))
            except Exception:
                rewards.append(0.0)
        else:
            rewards.append(1.0)
    return rewards

In [15]:
from trl import GRPOConfig

# Configure training arguments using GRPOConfig
training_args = GRPOConfig(
    output_dir="Qwen2-0.5B-GRPO-test",
    learning_rate=1e-5,
    remove_unused_columns=False,  # to access the solution column in accuracy_reward
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    bf16=True,
    # Parameters that control de data preprocessing
    max_completion_length=64,  # default: 256
    num_generations=4,  # default: 8
    # Parameters related to reporting and saving
    report_to=["tensorboard"],
    logging_steps=10,
    push_to_hub=False,
    save_strategy="steps",
    save_steps=10,
)

In [16]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/dlibland/miniforge3/envs/diffusion_rl/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Define the prompt using the required chat template
# The Qwen2-Instruct models use specific chat markers like <|im_start|> and <|im_end|>
prompt = "Describe a colonized alien planet that our spaceship is landing on. The description should have the feel of a travel guide."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate content
generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=512)
generated_ids = [
    output_ids[len(model_inputs.input_ids[0]) :] for output_ids in generated_ids
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response)

Welcome to the planet of Tera-1, where space and civilization meet. This place is a paradise for explorers, offering endless opportunities for discovery and adventure.

Our spaceship lands gently on the surface of this vast, sprawling world. The air is thick with the scent of fresh earth and the sound of rushing water. As we descend into the depths of the planet, the first thing you notice is the sheer size of it all. It's as big as a city, or even larger, depending on your perspective. The sky above is a canvas of stars, painted by the sun's rays, while the horizon stretches out like an endless sea of blue.

The land itself is a mix of rocky terrain and lush vegetation. The ground is covered in towering mountains, dotted with streams and rivers that flow like veins through the landscape. The trees are tall and majestic, their leaves rustling softly in the breeze. There are also dense forests of trees, some of which bear fruits that are as sweet as honey but more potent than any you've